## 查看fit_dualsg_all.json结构

In [1]:
import json
from pprint import pprint

path = "../fit_dualsg_all.json"

# 读取数据
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("总样本数:", len(data))
print("=" * 50)

# 看第1条
sample = data[0]
print("第1条数据结构:")
pprint(sample)

print("\n字段列表:")
print(list(sample.keys()))

print("=" * 50)

# 查看每个字段的类型
print("字段类型:")
for k, v in sample.items():
    print(f"{k}: {type(v)}")

print("=" * 50)

# 专门解析 annotations（你这个最关键）
ann = sample.get("annotations", None)

print("annotations 原始内容类型:", type(ann))

if isinstance(ann, str):
    ann_str = ann.strip()
    if ann_str.startswith("{"):
        try:
            ann_json = json.loads(ann_str)
            print("\nannotations 解析后:")
            pprint(ann_json)

            print("\nannotations 内字段:")
            print(list(ann_json.keys()))
        except Exception as e:
            print("annotations 解析失败:", e)
    else:
        print("annotations 是普通字符串:")
        print(ann_str[:200])

elif isinstance(ann, list):
    print("annotations 是 list，前几个元素:")
    print(ann[:5])

else:
    print("annotations 其他类型:", ann)

总样本数: 431592
第1条数据结构:
{'annotations': 'Fashion trend prediction for neckline:fur (attribute) in Hong '
                'Kong, targeting female consumers aged age_25_40. Historical '
                'popularity range: -0.004 to 0.031. The time series is '
                'relatively stable, with high volatility, peaking early in the '
                'period.',
 'metadata': {'element': 'neckline:fur',
              'group': 'age:age_25_40__city:Hong Kong__gender:female',
              'norm': [-0.003937726916966741, 0.03052326340898993, 0.01]},
 'series': [0.9881591253609898,
            0.8674461228041511,
            0.79150964590916,
            0.5844810964935376,
            0.5394748048614265,
            0.27962948919321257,
            0.21401679727599726,
            0.12426370769212049,
            0.2660648847900465,
            0.1306794967670219,
            0.2209046737623473,
            0.3623253175465997,
            0.7312980079842352,
            0.9206647719721105,
 

In [2]:
import os
import json
import time
from tqdm import tqdm
from openai import OpenAI

# ============================================================
# 0. 配置区（你只需要改这里）
# ============================================================
API_KEY = "sk-yqcegwvpdomtkrpnopynmdehgenmpzenjlaftxgxbcibqegv"
BASE_URL = "https://api.siliconflow.cn/v1"   # 例如: https://api.openai.com/v1
MODEL_NAME = "Qwen/Qwen3.5-9B"

INPUT_PATH = "../fit_dualsg_all.json"
OUTPUT_PATH = "../fit_dualsg_llm_batch10.jsonl"
ERROR_PATH = "../fit_dualsg_llm_batch10_error.jsonl"

START_IDX = 0
END_IDX = 30000   # 先跑3w条

BATCH_SIZE = 10
MAX_RETRIES = 5

# 是否断点续跑
RESUME = True


# ============================================================
# 1. 初始化模型
# ============================================================

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


# ============================================================
# 2. Prompt（JSON输出）
# ============================================================

SYSTEM_PROMPT = """
You are a time series analysis expert.

You will receive multiple time series samples.

For each sample, generate a semantic description with EXACTLY 5 lines:

Overall trend: ...
Volatility: ...
Turning points: ...
Temporal pattern: ...
Ending behavior: ...

Return ONLY valid JSON in the following format:

[
  {"id": 0, "annotation": "..."},
  {"id": 1, "annotation": "..."}
]

Rules:
- id must match input order (0 to N-1)
- no explanation
- no markdown
- no extra text
"""


# ============================================================
# 3. 工具函数
# ============================================================

def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def load_done_indices(path):
    done = set()
    if not os.path.exists(path):
        return done

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if "index" in obj:
                    done.add(obj["index"])
            except:
                continue
    return done


def build_batch_prompt(batch_series):
    text = "Analyze the following time series samples:\n\n"
    for i, s in enumerate(batch_series):
        text += f"Sample {i}: {json.dumps(s)}\n"
    return text


def clean_json_text(text):
    text = text.replace("```json", "").replace("```", "").strip()
    return text


def validate_batch_output(result, batch_size):
    if not isinstance(result, list):
        return False

    if len(result) != batch_size:
        return False

    ids = sorted([x.get("id") for x in result])
    return ids == list(range(batch_size))


# ============================================================
# 4. 主流程
# ============================================================

data = load_data(INPUT_PATH)

if END_IDX is None or END_IDX > len(data):
    END_IDX = len(data)

print("总样本数:", len(data))
print(f"处理范围: [{START_IDX}, {END_IDX})")

done_indices = set()
if RESUME:
    done_indices = load_done_indices(OUTPUT_PATH)
    print("已完成数量:", len(done_indices))

# 构造待处理索引
all_indices = list(range(START_IDX, END_IDX))
todo_indices = [i for i in all_indices if i not in done_indices]

print("待处理数量:", len(todo_indices))

# 按10条分组
batches = [
    todo_indices[i:i+BATCH_SIZE]
    for i in range(0, len(todo_indices), BATCH_SIZE)
]

print("总batch数:", len(batches))


# ============================================================
# 5. 开始跑
# ============================================================

for batch in tqdm(batches, desc="Batch进度"):

    # 如果最后一组不足10条，直接跳过（也可以选择处理）
    if len(batch) < BATCH_SIZE:
        continue

    batch_series = []
    for idx in batch:
        series = data[idx]["series"]
        batch_series.append(series)

    prompt = build_batch_prompt(batch_series)

    success = False
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                instructions=SYSTEM_PROMPT,
                input=prompt
            )

            text = response.output_text
            text = clean_json_text(text)

            result = json.loads(text)

            # 校验
            if not validate_batch_output(result, BATCH_SIZE):
                raise ValueError("JSON结构不符合要求")

            # 写入（映射回全局index）
            for item in result:
                local_id = item["id"]
                global_idx = batch[local_id]

                write_jsonl(OUTPUT_PATH, {
                    "index": global_idx,
                    "status": "ok",
                    "annotation": item["annotation"]
                })

            success = True
            break

        except Exception as e:
            last_error = str(e)
            time.sleep(2 * attempt)

    # 整组失败
    if not success:
        for idx in batch:
            write_jsonl(OUTPUT_PATH, {
                "index": idx,
                "status": "failed",
                "annotation": None,
                "error": last_error
            })
            write_jsonl(ERROR_PATH, {
                "index": idx,
                "error": last_error
            })

print("完成")

总样本数: 431592
处理范围: [0, 30000)
已完成数量: 0
待处理数量: 30000
总batch数: 3000


Batch进度:   0%|          | 10/3000 [06:00<29:55:52, 36.04s/it]


KeyboardInterrupt: 